# Goal and Work Summary

**Goal**: Build and evaluate a full-image semantic segmentation pipeline to detect defects in green rough oak planks (Background, BlackRot, Knot, Stain). The notebook focuses on cleaning/structuring the data, training a ResNet34‑UNet model, and producing reliable test‑set metrics and visual diagnostics.

## Workflow Stages and Rationale

**Stage 1 — Data audit and class normalization**
- We start by scanning the raw line‑scan dataset to quantify available samples and class masks.
- Duplicate or inconsistent defect names are normalized to a single class name to avoid label fragmentation.
- These steps are automated with custom helpers in `notebooks/functions` to ensure repeatability.

**Why this choice**: A consistent class taxonomy is required for stable training and honest evaluation; automated utilities reduce manual errors.

**Stage 2 — Filtering rare classes**
- Defect classes with very few masks are moved out to reduce class imbalance and training noise.
- The threshold is explicit so we can revisit it as more data arrives.

**Why this choice**: Extremely low‑frequency classes destabilize training and can dominate loss scaling for segmentation tasks.

**Stage 3 — Reorganization and validation**
- The dataset is reorganized into a clean structure with image/mask alignment.
- We validate integrity (missing masks, bad samples) and move invalid items to a filtered area.

**Why this choice**: Clean data organization simplifies dataloading and prevents silent misalignment during training.

**Stage 4 — Mask composition and splits**
- Separate defect masks are merged into a single multi‑class mask per image.
- Train/val/test splits are persisted to JSON for consistent experiment reuse.

**Why this choice**: Multi‑class masks match the model output and preserve consistent evaluation across runs.

**Stage 5 — Full‑image dataset + augmentations**
- A full‑image dataset class is defined with Albumentations transforms for robust training.
- Normalization uses ImageNet statistics to match the ResNet34 encoder initialization.

**Why this choice**: Full images preserve spatial context; augmentations reduce overfitting; ImageNet stats align with encoder priors.

**Stage 6 — Model training and checkpointing**
- We train a UNet with ResNet34 encoder using CrossEntropy loss.
- TensorBoard logs track loss and IoU; checkpoints are saved for reproducibility.

**Why this choice**: UNet is a strong baseline for segmentation; ResNet34 balances capacity and speed.

**Stage 7 — Best‑model selection and test evaluation**
- The best checkpoint is selected by validation IoU and saved to the models folder.
- The model is reloaded for test‑set inference to avoid training‑state leakage.
- Metrics include mean IoU, ROC curves (one‑vs‑rest), F1, accuracy, and a percentage confusion matrix.

**Why this choice**: Validation‑based selection provides a fair model for final reporting; multiple metrics reveal different failure modes.

## Custom Functions Used

We rely on project‑specific helpers (in `notebooks/functions`) to standardize data processing:
- `analyze_dataset`, `reorganize_dataset`, `validate_dataset`, `move_invalid_samples`
- `create_combined_masks`, `create_train_val_split`
- `rename_defect_files`, `filter_classes_by_mask_count`

These encapsulate repeatable steps and document the intended data workflow.

## References (Libraries)

| Library | Purpose | Notes | Link |
| --- | --- | --- | --- |
| PyTorch | Training, inference, dataloaders | Core deep learning framework and CUDA support | https://pytorch.org/ |
| segmentation_models_pytorch | UNet + encoder zoo | Provides ResNet34‑UNet and pretrained encoders | https://github.com/qubvel/segmentation_models.pytorch |
| Albumentations | Augmentations + normalization | Fast, flexible image transforms for segmentation | https://albumentations.ai/ |
| OpenCV | Image utilities | Used for file I/O and preprocessing support | https://opencv.org/ |
| NumPy | Array ops | Efficient tensor/array manipulation | https://numpy.org/ |
| Pillow | Image reading | Handles TIFF/PNG image loading | https://python-pillow.org/ |
| scikit‑learn | Metrics | ROC, F1, accuracy, confusion matrix | https://scikit-learn.org/ |
| Matplotlib | Plotting | Curves and confusion matrix visualization | https://matplotlib.org/ |
| TensorBoard | Logging | Visual tracking of training metrics | https://www.tensorflow.org/tensorboard |


In [ ]:
# Prepare Dataset
from functions.data_pipeline import prepare_dataset
import segmentation_models_pytorch as smp
from torch.utils.tensorboard import SummaryWriter
from functions.oak_hf_dataset import ensure_oak_raw_data

# Change this to use a custom folder name under data/
RAW_FOLDER_NAME = "raw-data"
RAW_DIR = ensure_oak_raw_data(raw_folder_name=RAW_FOLDER_NAME)

data_root, class_mapping = prepare_dataset(
    raw_data_folder=str(RAW_DIR),
    mask_threshold=200,     # minimum masks to keep a defect class
    val_split=0.16,
    test_split=0.2,
)
NUM_CLASSES = len(class_mapping) + 1  # +1 for Background

PREPARE_DATASET — SKIPPED (data already prepared)
  Organized data: ../data/organized-data
  Classes: {'BlackRot': 1, 'Knot': 2, 'Stain': 3}
  (pass force=True to re-run the full pipeline)


In [ ]:
# ──────────────────────────────────────────────
# Configuration - change these knobs as needed
# ──────────────────────────────────────────────
import torch

GPU_ID = 0  # choose 0 or 1

if torch.cuda.is_available():
    torch.cuda.set_device(GPU_ID)           # sets the current CUDA device
    DEVICE = torch.device(f"cuda:{GPU_ID}") # use this for .to(DEVICE)
    print(f"Using GPU {GPU_ID}: {torch.cuda.get_device_name(GPU_ID)}")
else:
    DEVICE = torch.device("cpu")
    print("Using CPU")

device = DEVICE  # alias used by training / evaluation cells below


In [2]:
from pathlib import Path
import cv2
import numpy as np
from PIL import Image
import json
import torch
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2

class FullImageSegmentationDataset(Dataset):
    def __init__(self, data_root, split='train', transform=None):
        self.data_root = Path(data_root)
        self.split = split
        self.transform = transform

        split_file = self.data_root / "split.json"
        with open(split_file, 'r') as f:
            split_info = json.load(f)
        self.image_ids = split_info[split]

        self.images_dir = self.data_root / "images"
        self.masks_dir = self.data_root / "combined_masks"

        print(f"Loaded {split} dataset: {len(self.image_ids)} images (full image mode)")

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_path = self.images_dir / f"{img_id}_Col.tif"
        mask_path = self.masks_dir / f"{img_id}_mask.png"
        image = np.array(Image.open(img_path).convert('RGB'))
        mask = np.array(Image.open(mask_path))

        if self.transform:
            transformed = self.transform(image=image, mask=mask)
            image = transformed['image']
            mask = transformed['mask']

        return image, mask.long()

# Full-image augmentations 
def get_training_augmentation_full():
    return A.Compose([
        A.Resize(512, 512),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.5),
        A.OneOf([A.RandomBrightnessContrast(p=1), A.RandomGamma(p=1)], p=0.3),
        A.OneOf([A.GaussNoise(p=1), A.GaussianBlur(p=1)], p=0.2),
        A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ToTensorV2()
    ])

def get_validation_augmentation_full():
    return A.Compose([
        A.Resize(512, 512),
        A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ToTensorV2()
    ])

def create_dataloaders_full(data_root, batch_size=2, num_workers=4):
    train_dataset = FullImageSegmentationDataset(
        data_root=data_root, split='train',
        transform=get_training_augmentation_full()
    )
    val_dataset = FullImageSegmentationDataset(
        data_root=data_root, split='val',
        transform=get_validation_augmentation_full()
    )
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader

In [3]:
# Use 224px patches and 50% overlap (stride=112). adjust batch sizes to fit GPU.
train_loader, val_loader = create_dataloaders_full(
    data_root="../data/organized-data",
    batch_size=32,          # adjust to GPU memory
    num_workers=4
)

#num_classes = len(class_mapping)  # or set explicitly
num_classes = 4  # set explicitly based on your dataset
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=num_classes
).to(DEVICE)



/home/vscode/.local/lib/python3.10/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


Loaded train dataset: 688 images (full image mode)
Loaded val dataset: 132 images (full image mode)


### Model Training

In [4]:
# CUDA device is selected in the Configuration cell (GPU_ID / DEVICE).

CUDA available: True
Current device: 0
Device name: NVIDIA GeForce RTX 3090


In [ ]:
import torch
import torch.nn as nn
from torch.optim import Adam
from pathlib import Path
import json

# Setup training (device set in Configuration cell above)
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=1e-3)
num_epochs = 50
checkpoint_dir = Path("../checkpoints/UNet_FullImage") # Change this to the model name
checkpoint_dir.mkdir(exist_ok=True)
best_model_path = checkpoint_dir / "best_model.pt"
best_val_iou = float("-inf")

# Training history
history = {
    'train_loss': [],
    'val_loss': [],
    'val_iou': []
}

def compute_iou(pred, target, num_classes):
    """Compute per-class IoU (Intersection over Union)."""
    ious = []
    for cls in range(num_classes):
        pred_mask = (pred == cls)
        target_mask = (target == cls)
        intersection = (pred_mask & target_mask).sum().float()
        union = (pred_mask | target_mask).sum().float()
        iou = intersection / (union + 1e-6)
        ious.append(iou.item())
    return np.mean(ious)

def train_epoch(model, train_loader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0.0
    num_batches = 0
    
    for img_patch, mask_patch in train_loader:
        img_patch = img_patch.to(device)
        mask_patch = mask_patch.to(device)
        
        # Forward
        logits = model(img_patch)  # [B, C, H, W]
        loss = criterion(logits, mask_patch)  # [B, H, W]
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
    
    return total_loss / num_batches

def validate(model, val_loader, criterion, device, num_classes):
    """Validate model."""
    model.eval()
    total_loss = 0.0
    total_iou = 0.0
    num_batches = 0
    
    with torch.no_grad():
        for img_patch, mask_patch in val_loader:
            img_patch = img_patch.to(device)
            mask_patch = mask_patch.to(device)
            
            logits = model(img_patch)
            loss = criterion(logits, mask_patch)
            pred = torch.argmax(logits, dim=1)
            
            iou = compute_iou(pred, mask_patch, num_classes)
            
            total_loss += loss.item()
            total_iou += iou
            num_batches += 1
    
    return total_loss / num_batches, total_iou / num_batches

# Create a TensorBoard writer
writer = SummaryWriter(log_dir="../logs/tensorboard/ResNet34_Unet_FullImage")

print("Using device:", device)
print("Model device:", next(model.parameters()).device)

# Training loop
for epoch in range(num_epochs):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_iou = validate(model, val_loader, criterion, device, num_classes)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_iou'].append(val_iou)
    
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val IoU: {val_iou:.4f}")
    
    # Log to TensorBoard
    writer.add_scalar("Loss/Train", train_loss, epoch)
    writer.add_scalar("Loss/Val", val_loss, epoch)
    writer.add_scalar("IoU/Val", val_iou, epoch)

    # Save best model whenever validation IoU improves
    if val_iou > best_val_iou:
        best_val_iou = val_iou
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_iou': val_iou,
        }, best_model_path)
        print(f"  -> New best model saved: {best_model_path} (val_iou={best_val_iou:.4f})")

    # Save checkpoint every 5 epochs
    if (epoch + 1) % 5 == 0:
        ckpt_path = checkpoint_dir / f"model_epoch_{epoch+1}.pt"
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_iou': val_iou,
        }, ckpt_path)
        print(f"  -> Checkpoint saved: {ckpt_path}")

# Save final history
history_path = checkpoint_dir / "training_history.json"
with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)
print(f"\nTraining complete. History saved to {history_path}")
print(f"Best model checkpoint: {best_model_path} (val_iou={best_val_iou:.4f})")

Using device: cuda
Model device: cuda:0


Epoch 1/50 | Train Loss: 1.1176 | Val Loss: 2.5666 | Val IoU: 0.2244
  -> New best model saved: ../checkpoints/UNet_FullImage/best_model.pt (val_iou=0.2244)
Epoch 2/50 | Train Loss: 0.4670 | Val Loss: 0.5196 | Val IoU: 0.2805
  -> New best model saved: ../checkpoints/UNet_FullImage/best_model.pt (val_iou=0.2805)
Epoch 3/50 | Train Loss: 0.3346 | Val Loss: 0.4310 | Val IoU: 0.3051
  -> New best model saved: ../checkpoints/UNet_FullImage/best_model.pt (val_iou=0.3051)
Epoch 4/50 | Train Loss: 0.3083 | Val Loss: 0.4043 | Val IoU: 0.3199
  -> New best model saved: ../checkpoints/UNet_FullImage/best_model.pt (val_iou=0.3199)
Epoch 5/50 | Train Loss: 0.2937 | Val Loss: 0.3141 | Val IoU: 0.3353
  -> New best model saved: ../checkpoints/UNet_FullImage/best_model.pt (val_iou=0.3353)
  -> Checkpoint saved: ../checkpoints/UNet_FullImage/model_epoch_5.pt
Epoch 6/50 | Train Loss: 0.2754 | Val Loss: 0.3550 | Val IoU: 0.2993
Epoch 7/50 | Train Loss: 0.2834 | Val Loss: 0.2818 | Val IoU: 0.3623
  -> Ne

In [ ]:
# Create test loader
test_dataset = FullImageSegmentationDataset(
    data_root="../data/organized-data",
    split="test",
    transform=get_validation_augmentation_full()
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=8, shuffle=False, num_workers=4, pin_memory=True
)

In [ ]:
import segmentation_models_pytorch as smp

model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights=None,  # weights already in state_dict
    in_channels=3,
    classes=num_classes
).to(device)

ckpt = torch.load(best_model_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

print(f"Loaded best model (epoch={ckpt.get('epoch')}, val_iou={ckpt.get('val_iou'):.4f})")

In [ ]:
import numpy as np
import torch
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score, roc_curve, auc
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

# class names (update if you change mapping)
class_names = ["Background", "BlackRot", "Knot", "Stain"]

all_preds = []
all_targets = []
all_probs = []

with torch.no_grad():
    for images, masks in test_loader:
        images = images.to(device)
        masks = masks.to(device)

        logits = model(images)  # [B,C,H,W]
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        all_preds.append(preds.cpu().numpy().reshape(-1))
        all_targets.append(masks.cpu().numpy().reshape(-1))
        all_probs.append(probs.cpu().numpy().transpose(0, 2, 3, 1).reshape(-1, num_classes))

all_preds = np.concatenate(all_preds)
all_targets = np.concatenate(all_targets)
all_probs = np.concatenate(all_probs)

# Confusion matrix (percentages by true class)
cm = confusion_matrix(all_targets, all_preds, labels=list(range(num_classes)))
cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)
cm_norm = np.nan_to_num(cm_norm)  # handle any empty classes

disp = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(ax=ax, cmap="Blues", values_format=".1%", colorbar=True, xticks_rotation=45)
plt.title("Confusion Matrix (Percent of True Class)")
plt.tight_layout()
plt.show()

# Accuracy (pixel-level)
acc = accuracy_score(all_targets, all_preds)
print(f"Accuracy: {acc:.4f}")

# F1 scores
f1_per_class = f1_score(all_targets, all_preds, labels=list(range(num_classes)), average=None)
f1_macro = f1_score(all_targets, all_preds, labels=list(range(num_classes)), average="macro")

print("F1 per class:")
for i, f1 in enumerate(f1_per_class):
    print(f"  {class_names[i]}: {f1:.4f}")
print(f"F1 Macro: {f1_macro:.4f}")

# IoU per class + mean IoU
intersection = np.diag(cm).astype(np.float32)
union = cm.sum(axis=0) + cm.sum(axis=1) - intersection
iou_per_class = np.divide(intersection, union, out=np.zeros_like(intersection), where=union != 0)

print("IoU per class:")
for i, iou in enumerate(iou_per_class):
    print(f"  {class_names[i]}: {iou:.4f}")

include_background = False
if include_background:
    mean_iou = iou_per_class.mean()
else:
    mean_iou = iou_per_class[1:].mean()  # skip background
print(f"Mean IoU ({'incl' if include_background else 'excl'} background): {mean_iou:.4f}")

# ROC curves (one-vs-rest)
plt.figure(figsize=(7, 5))
for c in range(num_classes):
    y_true_bin = (all_targets == c).astype(int)
    y_score = all_probs[:, c]

    # skip if class absent
    if y_true_bin.sum() == 0:
        print(f"ROC skipped: {class_names[c]} (no positives in test set)")
        continue

    fpr, tpr, _ = roc_curve(y_true_bin, y_score)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{class_names[c]} (AUC={roc_auc:.3f})")

plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves (One-vs-Rest)")
plt.legend()
plt.show()

In [ ]:
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.onnx
import numpy as np

# Figure out project root (works when running from notebooks/)
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


# ── EBI-Compliant Export Wrapper ──────────────────────
class EBIExportWrapper(nn.Module):
    """
    Wraps a trained segmentation model for EBI-compliant ONNX export.

    The ONNX graph will contain:
      Input:  uint8  [B, 3, H, W]  (raw RGB image, 0-255)
      Output: float32 [B, C, H, W] (per-pixel class scores, 0-100%)

    Baked-in operations:
      1. Cast uint8 → float32, divide by 255.0
      2. ImageNet mean/std normalization
      3. Forward through the segmentation backbone+decoder (logits)
      4. Softmax over the class dimension
      5. Multiply by 100.0 (percentage scale)
    """

    def __init__(self, seg_model, mean=None, std=None):
        super().__init__()
        self.seg_model = seg_model
        if mean is None:
            mean = [0.485, 0.456, 0.406]
        if std is None:
            std = [0.229, 0.224, 0.225]
        self.register_buffer("mean", torch.tensor(mean, dtype=torch.float32).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(std, dtype=torch.float32).view(1, 3, 1, 1))

    def forward(self, x):
        # x: uint8 [B, 3, H, W] → float32 [B, 3, H, W] in [0, 1]
        x = x.float() / 255.0
        # ImageNet normalization
        x = (x - self.mean) / self.std
        # Segmentation model forward
        logits = self.seg_model(x)
        # Softmax over class dimension → probabilities [0, 1]
        probs = F.softmax(logits, dim=1)
        # Scale to 0-100%
        scores = probs * 100.0
        return scores


# ── Build the EBI wrapper and export ──────────────────
device = next(model.parameters()).device
model.eval()

ebi_model = EBIExportWrapper(model).to(device)
ebi_model.eval()

# Where to save ONNX models
onnx_dir = project_root / "models" / "onnx"
onnx_dir.mkdir(parents=True, exist_ok=True)
onnx_path = onnx_dir / "unet_fullimage.onnx"

# Dummy input: uint8 image (as the ONNX graph expects raw images)
dummy_input = torch.randint(0, 256, (1, 3, 512, 512), dtype=torch.uint8, device=device)

torch.onnx.export(
    ebi_model,
    dummy_input,
    onnx_path.as_posix(),
    input_names=["input"],
    output_names=["output"],
    opset_version=12,
    dynamo=False,
    dynamic_axes={
        "input":  {0: "batch", 2: "height", 3: "width"},
        "output": {0: "batch", 2: "height", 3: "width"},
    },
)

print(f"✓ Exported EBI-compliant ONNX model to: {onnx_path}")
print(f"  Input:  uint8  [B, 3, H, W]  (raw RGB 0-255)")
print(f"  Output: float32 [B, C, H, W] (per-pixel class scores 0-100%)")

# ── Quick verification ────────────────────────────────
import onnxruntime as ort

session = ort.InferenceSession(onnx_path.as_posix())
inp_meta = session.get_inputs()[0]
out_meta = session.get_outputs()[0]
print(f"\nONNX Input:  name={inp_meta.name}, shape={inp_meta.shape}, type={inp_meta.type}")
print(f"ONNX Output: name={out_meta.name}, shape={out_meta.shape}, type={out_meta.type}")

test_input = np.random.randint(0, 256, (1, 3, 512, 512)).astype(np.uint8)
onnx_output = session.run(None, {inp_meta.name: test_input})[0]
print(f"  Output range: [{onnx_output.min():.2f}, {onnx_output.max():.2f}]")
print(f"  Sum along class axis: min={onnx_output.sum(axis=1).min():.2f}, max={onnx_output.sum(axis=1).max():.2f}")
print("✓ ONNX verification passed")

In [ ]:
from pathlib import Path
import sys
import json

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
notebooks_dir = project_root / "notebooks"
if str(notebooks_dir) not in sys.path:
    sys.path.append(str(notebooks_dir))

from functions.metrics_export import (
    append_row_to_csv,
    build_run_row,
    first_available,
    guess_actual_epochs,
    infer_split_sizes,
)

local_vars = locals()

history = local_vars.get("history")
training_history_path = local_vars.get("history_path")
if training_history_path is None and "checkpoint_dir" in local_vars:
    training_history_path = local_vars["checkpoint_dir"] / "training_history.json"
if training_history_path is None and "CHECKPOINT_DIR" in local_vars:
    training_history_path = local_vars["CHECKPOINT_DIR"] / "training_history.json"
if history is None and training_history_path is not None and Path(training_history_path).exists():
    with open(training_history_path, "r", encoding="utf-8") as f:
        history = json.load(f)

train_split_size, val_split_size, test_split_size = infer_split_sizes(project_root)

metrics_dir = Path(local_vars.get("METRICS_DIR", project_root / "data" / "metrics"))
csv_path = metrics_dir / "TRAINING_RESULTS_SUMMARY.csv"

extra_training_params = {}
for key in [
    "ENCODER_NAME",
    "NUM_EPOCHS",
    "BATCH_SIZE",
    "NUM_WORKERS",
    "LEARNING_RATE",
    "WEIGHT_DECAY",
    "PATIENCE",
    "IMAGE_HEIGHT",
    "IMAGE_WIDTH",
]:
    if key in local_vars:
        extra_training_params[key] = local_vars[key]

row = build_run_row(
    notebook_name="Unet_FullImage",
    model_name=str(first_available(local_vars, ["MODEL_ARCH", "CHECKPOINT_NAME"]) or "UNet"),
    raw_data_folder=first_available(local_vars, ["RAW_DIR"]),
    image_height=first_available(local_vars, ["IMAGE_HEIGHT"]),
    image_width=first_available(local_vars, ["IMAGE_WIDTH"]),
    batch_size=first_available(local_vars, ["BATCH_SIZE", "batch_size"]),
    num_workers=first_available(local_vars, ["NUM_WORKERS", "num_workers"]),
    num_epochs_config=first_available(local_vars, ["NUM_EPOCHS", "num_epochs"]),
    num_epochs_actual=guess_actual_epochs(history),
    optimizer_name=first_available(local_vars, ["OPTIMIZER"]) or "Adam",
    learning_rate=first_available(local_vars, ["LEARNING_RATE"]),
    weight_decay=first_available(local_vars, ["WEIGHT_DECAY"]),
    patience=first_available(local_vars, ["PATIENCE"]),
    loss_name="CrossEntropyLoss",
    num_classes=first_available(local_vars, ["NUM_CLASSES", "num_classes"]),
    class_names=first_available(local_vars, ["CLASS_NAMES", "class_names"]),
    train_split_size=train_split_size,
    val_split_size=val_split_size,
    test_split_size=test_split_size,
    accuracy=first_available(local_vars, ["acc", "accuracy", "pixel_acc_full", "pixel_acc"]),
    f1_macro=first_available(local_vars, ["f1_macro"]),
    f1_per_class=first_available(local_vars, ["f1_per_class"]),
    mean_iou=first_available(local_vars, ["mean_iou_excl_bg", "mean_iou"]),
    iou_per_class=first_available(local_vars, ["iou_per_class", "test_ious"]),
    checkpoint_path=first_available(local_vars, ["best_model_path", "deployment_checkpoint"]),
    training_history_path=training_history_path,
    onnx_path=first_available(local_vars, ["onnx_path"]),
    run_name=first_available(local_vars, ["METRICS_RUN_NAME", "CHECKPOINT_NAME"]),
    transfer_learning=first_available(local_vars, ["TRANSFER_LEARNING"]),
    transfer_checkpoint_path=first_available(local_vars, ["TRANSFER_CHECKPOINT_PATH"]),
    freeze_encoder_epochs=first_available(local_vars, ["FREEZE_ENCODER_EPOCHS"]),
    optuna_enabled=False,
    extra_training_params=extra_training_params,
)

written_path = append_row_to_csv(csv_path, row)
print(f"Appended metrics row to: {written_path}")